In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle
from sklearn.metrics import brier_score_loss, make_scorer, log_loss
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import shap

In [125]:
df = pd.read_csv("../Data/cleanedHealthData.csv")

In [126]:
## Establish weights for target variable
healthy_weight = df['target'].value_counts(normalize=True)['healthy']
diseased_weight = df['target'].value_counts(normalize=True)['diseased']

In [127]:
## Separate all the columns
numerical_cols = df.select_dtypes(include=['number']).columns
# numerical_cols = [col for col in numerical_cols if col != 'target']

categorical_cols = list(df.select_dtypes(exclude=['number']).columns)
categorical_cols = [col for col in categorical_cols if col != 'target']


In [128]:
## Convert object columns to categorical
obj_cols = df.select_dtypes(include=["object"]).columns
df[obj_cols] = df[obj_cols].astype("category")

In [129]:
## Set target as 1|0

df['target'] = df['target'].replace({'healthy': 0, 'diseased': 1})

y = df['target']
X = df.drop(columns='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

C:\Users\admin\AppData\Local\Temp\ipykernel_31312\3429725909.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['target'] = df['target'].replace({'healthy': 0, 'diseased': 1})
C:\Users\admin\AppData\Local\Temp\ipykernel_31312\3429725909.py:3: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['target'] = df['target'].replace({'healthy': 0, 'diseased': 1})


In [ ]:
## Setup Numerical and Categorical Cols Transformers (KNNImputer, Scaler and OHE)

num_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=3)), ## Reduce to 3 to save computation
    ('scaler', StandardScaler(with_mean=False)),
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe', OneHotEncoder(drop = 'first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

## For Random Forest only 
imp_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=3))
])

## XGB category columns
category_transformer = FunctionTransformer(lambda df: df.astype('category'))

xgb_preprocessor = ColumnTransformer(
    transformers = [
        ('num', imp_transformer, numerical_cols),
        ('cat', category_transformer, categorical_cols)
    ])

rf_preprocesser = ColumnTransformer(
    transformers = [
        ('num', imp_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ] 
)

## Baseline Logistic Regression Model

In [133]:
## Logistic Regression pipeline (0.210771)

base_models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
}

results = []
for model, pipeline in tqdm(base_models.items(), desc="Training Models"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

Training Models: 100%|██████████| 1/1 [04:11<00:00, 251.51s/it]


,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210762,0.612561


In [ ]:
def get_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = []

    if 'num' in preprocessor.named_transformers_:
        feature_names.extend(numeric_cols)

    if 'cat' in preprocessor.named_transformers_:
        ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
        ohe_names = ohe.get_feature_names_out(categorical_cols)
        feature_names.extend(ohe_names)

    return feature_names

## Use LASSO, Random Forest, KMeans for Feature Selection

In [ ]:
## LASSO Feature Selection
fs_models = {
    'Lasso': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(penalty='l1', solver='liblinear', random_state=42))
    ])
}

results = []
for model, pipeline in tqdm(fs_models.items(), desc="Lasso Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    ## Get feature names (OHE produced more)
    pre = pipeline.named_steps['preprocessor']
    feature_names = get_feature_names(pre, numerical_cols, categorical_cols)
    
    lasso_importance = pd.Series(np.abs(pipeline['model'].coef_).flatten(), index=feature_names)
    lasso_features = set(lasso_importance[lasso_importance > 0].index)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df


Lasso Feature Selection: 100%|██████████| 1/1 [03:01<00:00, 181.63s/it]


,Model,Brier Score Loss,Log Loss
0,Lasso,0.210762,0.612562


In [ ]:
print(f"Lasso Feature Selection:\n{lasso_features}")

Lasso Feature Selection:
{'exercise_type_Strength', 'cholesterol', 'sugar_intake', 'blood_pressure', 'device_usage_Moderate', 'mental_health_support_Yes', 'family_history_Yes', 'smoking_level_Light', 'meals_per_day', 'screen_time', 'waist_size', 'alcohol_consumption_Regularly', 'daily_steps', 'height', 'job_type_Office', 'diet_type_Vegetarian', 'heart_rate', 'caffeine_intake_Missing', 'occupation_Farmer', 'physical_activity', 'work_hours', 'occupation_Engineer', 'job_type_Tech', 'job_type_Labor', 'exercise_type_Missing', 'education_level_Master', 'pet_owner_Yes', 'caffeine_intake_Moderate', 'sleep_quality_Poor', 'water_intake', 'education_level_High School', 'diet_type_Vegan', 'smoking_level_Non-smoker', 'gender_Male', 'insurance_Yes', 'occupation_Doctor', 'diet_type_Omnivore', 'occupation_Driver', 'calorie_intake', 'daily_supplement_dosage', 'healthcare_access_Poor', 'sleep_hours', 'sleep_quality_Good', 'weight', 'income', 'education_level_PhD', 'device_usage_Low', 'healthcare_access_

In [ ]:
## Random Forest Feature Selection
fs_models = {
    'RandomForest': Pipeline([
        ('preprocessor', rf_preprocesser),  ## No StandardScaler for RF
        ('model', RandomForestClassifier(random_state=42))
    ])
}

results = []
for model, pipeline in tqdm(fs_models.items(), desc="Random Forest Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    ## Get feature names (OHE produced more)
    pre = pipeline.named_steps['preprocessor']
    feature_names = get_feature_names(pre, numerical_cols, categorical_cols)
    
    rf_importance = pd.Series(pipeline['model'].feature_importances_, index=feature_names)
    rf_features = set(rf_importance[rf_importance > rf_importance.mean()].index)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

Random Forest Feature Selection: 100%|██████████| 1/1 [03:36<00:00, 216.60s/it]


,Model,Brier Score Loss,Log Loss
0,RandomForest,0.21366,0.619371


In [ ]:
rf_importance.sort_values(ascending=False)

insulin                     0.038739
heart_rate                  0.038703
waist_size                  0.038585
sugar_intake                0.038533
daily_steps                 0.038463
                              ...   
occupation_Farmer           0.004096
occupation_Doctor           0.004080
occupation_Engineer         0.003997
job_type_Office             0.003881
gene_marker_flag_Missing    0.003460
Length: 63, dtype: float64

In [ ]:
print(f"Random Forest Feature Selection:\n{rf_features}")

Random Forest Feature Selection:
{'cholesterol', 'sugar_intake', 'blood_pressure', 'screen_time', 'waist_size', 'daily_steps', 'height', 'heart_rate', 'physical_activity', 'work_hours', 'water_intake', 'calorie_intake', 'daily_supplement_dosage', 'sleep_hours', 'weight', 'income', 'mental_health_score', 'age', 'glucose', 'bmi_corrected', 'stress_level', 'insulin'}


In [ ]:
## XGB Feature Selection
fs_models = {
    'XGB': Pipeline([
        ('model', XGBClassifier(enable_categorical=True, random_state=42))   ## No scaling needed
    ])
}

results = []
for model, pipeline in tqdm(fs_models.items(), desc="XGB Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    explainer = shap.TreeExplainer(pipeline['model'])
    shap_values = explainer.shap_values(X_train)
   
    
    shap_importance = pd.Series(np.abs(shap_values).mean(0), index=X.columns)
    xgb_feature_names = set(shap_importance.nlargest(30).index)

    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

XGB Feature Selection:   0%|          | 0/1 [00:00<?, ?it/s]

XGB Feature Selection: 100%|██████████| 1/1 [00:06<00:00,  6.22s/it]


,Model,Brier Score Loss,Log Loss
0,XGB,0.218409,0.632276


In [ ]:
print(f"XGB Feature Selection:\n{xgb_feature_names}")

XGB Feature Selection:
{'cholesterol', 'exercise_type', 'sugar_intake', 'blood_pressure', 'occupation', 'sleep_quality', 'screen_time', 'waist_size', 'height', 'daily_steps', 'heart_rate', 'physical_activity', 'work_hours', 'job_type', 'water_intake', 'diet_type', 'calorie_intake', 'daily_supplement_dosage', 'sleep_hours', 'weight', 'income', 'mental_health_score', 'age', 'glucose', 'bmi_corrected', 'stress_level', 'education_level', 'healthcare_access', 'device_usage', 'insulin'}


In [ ]:
## Filtered out 22 features (length of rf_features)

selected_features = (
    lasso_features & rf_features |
    lasso_features & xgb_feature_names |
    rf_features & xgb_feature_names
)

selected_features = list(selected_features)
print(len(selected_features))
print("\n Final Selected Features:\n", selected_features)

22

 Final Selected Features:
 ['cholesterol', 'sugar_intake', 'blood_pressure', 'screen_time', 'waist_size', 'height', 'daily_steps', 'heart_rate', 'physical_activity', 'work_hours', 'water_intake', 'calorie_intake', 'daily_supplement_dosage', 'sleep_hours', 'weight', 'income', 'mental_health_score', 'age', 'glucose', 'bmi_corrected', 'stress_level', 'insulin']


## Baseline Pipeline of all Models on Full Data

In [ ]:
models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', rf_preprocesser),
        ('model', RandomForestClassifier(random_state=42))
    ]),
    'GradientBoosting': Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'KNN': Pipeline([
        ('preprocessor', preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'XGBoost': Pipeline([
        ('model', XGBClassifier(enable_categorical=True, random_state=42))
    ])
}

In [ ]:
results = []

for model, pipeline in tqdm(models.items(), desc="Training Baseline Models Full Data"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

baseline_results_df = pd.DataFrame(results).sort_values(by='Log Loss')
# results_df.to_csv('../Data/preliminaryResults.csv', index=False)
# results_df

Training Baseline Models Full Data: 100%|██████████| 6/6 [17:17<00:00, 172.99s/it]


In [ ]:
baseline_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210762,0.612561
2,GradientBoosting,0.210801,0.612651
1,RandomForest,0.213660,0.619371
5,XGBoost,0.218409,0.632276
4,KNN,0.251964,2.432118
3,DecisionTree,0.433250,15.615913


## Use Logistic Regression, Random Forest, Boosting Models to Evaluate Selection

In [155]:
## Filter dataset for selected features
# X_train = X_train[list(selected_features)]
# X_test = X_test[list(selected_features)]

selected_categorical_cols = [feature for feature in categorical_cols if feature in selected_features]
selected_numerical_cols = [feature for feature in numerical_cols if feature in selected_features]

## Adjust Preprocessor and Pipeline based on limited features

sel_preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, selected_numerical_cols),
        ('cat', cat_transformer, selected_categorical_cols),
    ], remainder='drop'
)

sel_rf_preprocesser = ColumnTransformer(
    transformers = [
        ('num', imp_transformer, selected_numerical_cols),
        ('cat', cat_transformer, selected_categorical_cols),
    ], remainder='drop'
)

sel_xgb_preprocessor = ColumnTransformer(
    transformers = [
        ('num', imp_transformer, selected_numerical_cols),
        ('cat', 'passthrough', selected_categorical_cols),
    ], remainder='drop'
)

sel_models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', sel_rf_preprocesser),
        ('model', RandomForestClassifier(random_state=42))
    ]),
    'GradientBoosting': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'KNN': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', sel_xgb_preprocessor),
        ('model', XGBClassifier(enable_categorical=True, random_state=42))
    ])
}


In [ ]:
results = []

for model, pipeline in tqdm(sel_models.items(), desc="Training Baseline Models Selected Data"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

selected_results_df = pd.DataFrame(results).sort_values(by='Log Loss')
# results_df.to_csv('../Data/preliminaryResults.csv', index=False)
# results_df

Training Baseline Models Selected Data: 100%|██████████| 6/6 [16:23<00:00, 163.95s/it]


In [ ]:
baseline_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210762,0.612561
2,GradientBoosting,0.210801,0.612651
1,RandomForest,0.213660,0.619371
5,XGBoost,0.218409,0.632276
4,KNN,0.251964,2.432118
3,DecisionTree,0.433250,15.615913


In [ ]:
selected_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210602,0.612176
2,GradientBoosting,0.210727,0.612485
1,RandomForest,0.213666,0.619433
5,XGBoost,0.218409,0.632276
4,KNN,0.251318,2.494181
3,DecisionTree,0.424650,15.305937


## Hyperparameter Tuning

In [ ]:
## Set parameters for each model
neg_class = y.value_counts()[0]
pos_class = y.value_counts()[1]
ratio = neg_class / pos_class

param_grids = {
    "GradientBoosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [3, 4, 5],
        "model__subsample": [0.6, 0.8, 1.0],
    },
    "LogisticRegression": {
        "model__C": np.logspace(-3, 3, 10),
        "model__solver": ["liblinear", "saga"],
        "model__penalty": ["l2", "l1"],
        "model__class_weight": ['balanced', None]
    },
    "RandomForest": {
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [None, 5, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__class_weight": ['balanced', None]

    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 13, 15],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    },
    "DecisionTree": {
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__criterion": ["gini", "entropy"],
        "model__class_weight": ['balanced', None]
    },
    'XGBoost': {
        "model__n_estimators": [200, 400, 800],
        "model__learning_rate": [0.001, 0.05, 0.1],
        "model__max_depth": [2, 3, 4, 5, 6],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__scale_pos_weight": [1, ratio*0.5, ratio, ratio*2]  ## Negative Class / Positive Class
    }
    
}

In [ ]:
log_loss_scorer = make_scorer(log_loss, response_method='predict_proba', greater_is_better=False, labels=[0,1])
brier_scorer = make_scorer(brier_score_loss, response_method='predict_proba', greater_is_better=False, labels=[0,1])

scoring = {
    "log_loss": log_loss_scorer,
    "brier": brier_scorer
}

In [ ]:
## Features that were not selected
features_notUsed = list(set(X.columns).difference(set(selected_features)))
sorted(features_notUsed)

['alcohol_consumption',
 'caffeine_intake',
 'device_usage',
 'diet_type',
 'education_level',
 'exercise_type',
 'family_history',
 'gender',
 'gene_marker_flag',
 'healthcare_access',
 'insurance',
 'job_type',
 'meals_per_day',
 'mental_health_support',
 'occupation',
 'pet_owner',
 'sleep_quality',
 'smoking_level',
 'sunlight_exposure']

In [ ]:
## Perform Randomized Search on selected features
fs_best_models = {}

## Need to use selected features models pipeline
for name, pipeline in sel_models.items():
    print(f"\n Running RandomizedSearchCV for {name}...")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grids[name],
        n_iter=10,              ## try 10 random combinations
        scoring=scoring,
        refit='log_loss', 
        cv=3,                   ## reduce to increase computation speed
        verbose=2,
        random_state=42,
        error_score='raise',
        n_jobs=4,               ## use 4 CPU cores
        pre_dispatch="2*n_jobs"
    )
    search.fit(X_train, y_train)
    fs_best_models[name] = search.best_estimator_
    
    print(f"Best params for {name}: {search.best_params_}")
    print(f"Best CV log loss: {-search.best_score_:.4f}")


## Evaluate best models on test set
print("\n Test Log Loss:")
models_dict = {}
for name, model in fs_best_models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    ll = log_loss(y_test, y_pred_proba)
    models_dict[name]= ll
    print(f"{name}: {ll:.4f}")

    
## Pickle best model
with open("selectedFeatures_best_model.pkl", "wb") as f:
    pickle.dump(fs_best_models[name], f)

res_df = pd.DataFrame.from_dict(models_dict, orient='index').reset_index()
res_df.columns = ['Model', 'Test Log Loss']
res_df = res_df.sort_values(by='Test Log Loss', ascending=True)
res_df.to_csv("../Data/tunedFSAllModels_new.csv", index=False)


 Running RandomizedSearchCV for LogisticRegression...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for LogisticRegression: {'model__solver': 'liblinear', 'model__penalty': 'l1', 'model__class_weight': None, 'model__C': np.float64(0.021544346900318832)}
Best CV log loss: 0.6097

 Running RandomizedSearchCV for RandomForest...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for RandomForest: {'model__n_estimators': 100, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_depth': 5, 'model__class_weight': None}
Best CV log loss: 0.6097

 Running RandomizedSearchCV for GradientBoosting...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for GradientBoosting: {'model__subsample': 0.6, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01}
Best CV log loss: 0.6096

 Running RandomizedSearchCV for DecisionTree...
Fitting 3 folds for each of 10 candidates, totalling 30 

In [136]:
fs_best_params = {
    'LogisticRegression': {'model__solver': 'liblinear', 'model__penalty': 'l1', 'model__class_weight': None, 'model__C': np.float64(0.021544346900318832)},
    'RandomForest': {'model__n_estimators': 100, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_depth': 5, 'model__class_weight': None},
    'GradientBoosting': {'model__subsample': 0.6, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01},
    'DecisionTree': {'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_depth': 5, 'model__criterion': 'entropy', 'model__class_weight': None},
    'KNN': {'model__weights': 'distance', 'model__n_neighbors': 15, 'model__metric': 'euclidean'},
    'XGBoost': {'model__subsample': 1.0, 'model__scale_pos_weight': 1, 'model__n_estimators': 400, 'model__max_depth': 6, 'model__learning_rate': 0.001, 'model__colsample_bytree': 1.0}    
}

In [137]:
## Functions to get model coef or feature importance

def get_feature_names(pipeline):
    # pre = pipeline.named_steps['preprocessor']
    # return pre.get_feature_names_out()

    if "preprocessor" in pipeline.named_steps:
        pre = pipeline.named_steps["preprocessor"]
        return pre.get_feature_names_out()

    if X_train is not None:
        return X_train.columns.tolist()

    raise ValueError(
        "Pipeline has no preprocessor. Provide X_train to extract feature names."
    )

def get_model_figures(model, feature_names):
    clf = model.named_steps['model']
    
    if hasattr(clf, "coef_"):
        importance = clf.coef_.ravel()
    
    elif hasattr(clf, 'feature_importances_'):
        importance = clf.feature_importances_
    
    else:
        print("No coefficients or importances available for:", clf.__class__.__name__)
    
    return pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)

In [138]:
## Feature Selected Final Models

fs_final_models = {}

for name, search in sel_models.items():
    print(f"Refitting {name} with best parameters ")
    
    # best_params = search.best_params_
    ## Hardcoded Params for quicker turnaround after testing previously
    model = sel_models[name].set_params(**fs_best_params[name])
    model.fit(X_train, y_train)
    fs_final_models[name] = model

## Get feature importance for all models

fs_feature_importance_results = {}

for name, model in fs_final_models.items():
    print(f"\n Extracting importances for {name} ")
    
    feature_names = get_feature_names(model)
    try:
        importance_df = get_model_figures(model, feature_names)
    except:
        print(f"{name} does not have feature importance")
    
    fs_feature_importance_results[name] = importance_df

Refitting LogisticRegression with best parameters 
Refitting RandomForest with best parameters 
Refitting GradientBoosting with best parameters 
Refitting DecisionTree with best parameters 
Refitting KNN with best parameters 
Refitting XGBoost with best parameters 

 Extracting importances for LogisticRegression 

 Extracting importances for RandomForest 

 Extracting importances for GradientBoosting 

 Extracting importances for DecisionTree 

 Extracting importances for KNN 
No coefficients or importances available for: KNeighborsClassifier
KNN does not have feature importance

 Extracting importances for XGBoost 


In [140]:
## View Feature Selected Feature Coefficients/Importances

with pd.ExcelWriter("../Data/selectedFeaturesCoefs", engine='xlsxwriter') as writer:
    for name, df in fs_feature_importance_results.items():
        print(f"\n Model: {name}")
        print(df.head())
        df.to_excel(writer, sheet_name = name, index=False)

    


 Model: LogisticRegression
                     feature  importance
2                num__weight    0.049031
19  num__mental_health_score    0.007901
0                   num__age    0.004525
9               num__insulin    0.002236
12    num__physical_activity    0.001084

 Model: RandomForest
             feature  importance
11   num__work_hours    0.075671
7   num__cholesterol    0.062896
6    num__heart_rate    0.062801
4    num__waist_size    0.057791
9       num__insulin    0.057676

 Model: GradientBoosting
             feature  importance
11   num__work_hours    0.095191
6    num__heart_rate    0.087648
7   num__cholesterol    0.076308
1        num__height    0.064366
20       num__income    0.063739

 Model: DecisionTree
                feature  importance
11      num__work_hours    0.137690
15    num__sugar_intake    0.137105
8          num__glucose    0.121395
13     num__daily_steps    0.116394
5   num__blood_pressure    0.110009

 Model: KNN
                feature  import

In [161]:
## Perform Randomized Search on all features
all_best_models = {}

for name, pipeline in models.items():
    print(f"\n Running RandomizedSearchCV for {name}...")
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_grids[name],
        n_iter=10,              ## try 10 random combinations
        scoring=scoring,
        refit='log_loss',  
        cv=3,                   ## reduce to increase computation speed
        verbose=2,
        random_state=42,
        error_score='raise',
        n_jobs=4,               ## use 4 CPU cores
        pre_dispatch="2*n_jobs"              
    )
    search.fit(X_train, y_train)
    all_best_models[name] = search.best_estimator_
    
    print(f"Best params for {name}: {search.best_params_}")
    print(f"Best CV log loss: {-search.best_score_:.4f}")

models_dict = {}
# Evaluate best models on test set
print("\n Test Log Loss:")
for name, model in all_best_models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    ll = log_loss(y_test, y_pred_proba)
    models_dict[name]= ll
    print(f"{name}: {ll:.4f}")

## Pickle best model
with open("allFeatures_best_model.pkl", "wb") as f:
    pickle.dump(all_best_models[name], f)

res_df = pd.DataFrame.from_dict(models_dict, orient='index').reset_index()
res_df.columns = ['Model', 'Test Log Loss']
res_df = res_df.sort_values(by='Test Log Loss', ascending=True)
res_df.to_csv("../Data/tunedFullAllModels_new.csv", index=False)


 Running RandomizedSearchCV for LogisticRegression...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for LogisticRegression: {'model__solver': 'liblinear', 'model__penalty': 'l1', 'model__class_weight': None, 'model__C': np.float64(0.021544346900318832)}
Best CV log loss: 0.6097

 Running RandomizedSearchCV for RandomForest...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for RandomForest: {'model__n_estimators': 100, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_depth': 5, 'model__class_weight': None}
Best CV log loss: 0.6096

 Running RandomizedSearchCV for GradientBoosting...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params for GradientBoosting: {'model__subsample': 0.6, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01}
Best CV log loss: 0.6096

 Running RandomizedSearchCV for DecisionTree...
Fitting 3 folds for each of 10 candidates, totalling 30 

In [153]:
all_best_params = {
    'LogisticRegression': {'model__solver': 'liblinear', 'model__penalty': 'l1', 'model__class_weight': None, 'model__C': np.float64(0.021544346900318832)},
    'RandomForest': {'model__n_estimators': 100, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_depth': 5, 'model__class_weight': None},
    'GradientBoosting': {'model__subsample': 0.6, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01},
    'DecisionTree': {'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_depth': 5, 'model__criterion': 'gini', 'model__class_weight': None},
    'KNN': {'model__weights': 'distance', 'model__n_neighbors': 15, 'model__metric': 'euclidean'},
    'XGBoost': {'model__subsample': 1.0, 'model__scale_pos_weight': 1, 'model__n_estimators': 400, 'model__max_depth': 6, 'model__learning_rate': 0.001, 'model__colsample_bytree': 1.0}  
}

In [159]:
## All Final Models

all_final_models = {}

for name, search in models.items():
    print(f"Refitting {name} with best parameters ")
    
    # best_params = search.best_params_
    ## Hardcoded Params for quicker turnaround after testing previously
    model = models[name].set_params(**all_best_params[name])
    model.fit(X_train, y_train)
    all_final_models[name] = model

## Get feature importance for all models

all_feature_importance_results = {}

for name, model in all_final_models.items():
    print(f"\n Extracting importances for {name} ")
    
    feature_names = get_feature_names(model)
    try:
        importance_df = get_model_figures(model, feature_names)
    except:
        print(f"{name} does not have feature importance")
    
    all_feature_importance_results[name] = importance_df

Refitting LogisticRegression with best parameters 
Refitting RandomForest with best parameters 
Refitting GradientBoosting with best parameters 
Refitting DecisionTree with best parameters 
Refitting KNN with best parameters 
Refitting XGBoost with best parameters 

 Extracting importances for LogisticRegression 

 Extracting importances for RandomForest 

 Extracting importances for GradientBoosting 

 Extracting importances for DecisionTree 

 Extracting importances for KNN 
No coefficients or importances available for: KNeighborsClassifier
KNN does not have feature importance

 Extracting importances for XGBoost 


In [160]:
## View Feature Selected Feature Coefficients/Importances

with pd.ExcelWriter("../Data/allFeaturesCoefs", engine='xlsxwriter') as writer:
    for name, df in all_feature_importance_results.items():
        print(f"\n Model: {name}")
        print(df.head())
        df.to_excel(writer, sheet_name = name, index=False)




 Model: LogisticRegression
                          feature  importance
2                     num__weight    0.046737
59  cat__caffeine_intake_Moderate    0.009869
19       num__mental_health_score    0.008019
43         cat__occupation_Farmer    0.006030
0                        num__age    0.004658

 Model: RandomForest
              feature  importance
11    num__work_hours    0.058183
6     num__heart_rate    0.053573
8        num__glucose    0.050311
4     num__waist_size    0.049892
16  num__water_intake    0.049445

 Model: GradientBoosting
             feature  importance
11   num__work_hours    0.090262
6    num__heart_rate    0.086568
7   num__cholesterol    0.074460
20       num__income    0.062116
1        num__height    0.060195

 Model: DecisionTree
               feature  importance
15   num__sugar_intake    0.213579
6      num__heart_rate    0.157009
7     num__cholesterol    0.100913
8         num__glucose    0.096282
3   num__bmi_corrected    0.089608

 Model: KNN
 